# Retail Sales Performance & Profitability Analysis

**Business question:** Which product categories and regions drive sales and profit,
and how can a retail company optimize discounting and shipping strategies to improve profitability?

**Dataset:** Superstore sales dataset (public, 8,399 line items).

This notebook cleans the data, computes profitability KPIs, produces charts and
exports the numbers consumed by the dashboard (`docs/dashboard.json`).

In [1]:
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA = ROOT / "data" / "superstore.csv"
OUT_CHARTS = ROOT / "outputs" / "charts"
OUT_DOCS = ROOT / "docs" / "data"
OUT_CHARTS.mkdir(parents=True, exist_ok=True)
OUT_DOCS.mkdir(parents=True, exist_ok=True)
(ROOT / "outputs").mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})
PALETTE = ["#1f4e79", "#2e75b6", "#9dc3e6", "#c55a11", "#f4b183"]

## 1. Load the data

In [2]:
df = pd.read_csv(DATA)
print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
df.head()

Rows: 8,399 | Columns: 21


,Row ID,Order ID,Order Date,Order Priority,Order Quantity,Sales,Discount,Ship Mode,Profit,Unit Price,...,Customer Name,Province,Region,Customer Segment,Product Category,Product Sub-Category,Product Name,Product Container,Product Base Margin,Ship Date
0,1,3,10/13/2010,Low,6,261.5400,0.04,Regular Air,-213.25,38.94,...,Muhammed MacIntyre,Nunavut,Nunavut,Small Business,Office Supplies,Storage & Organization,"Eldon Base for stackable storage shelf, platinum",Large Box,0.80,10/20/2010
1,49,293,10/1/2012,High,49,10123.0200,0.07,Delivery Truck,457.81,208.16,...,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,"1.7 Cubic Foot Compact ""Cube"" Office Refrigera...",Jumbo Drum,0.58,10/2/2012
2,50,293,10/1/2012,High,27,244.5700,0.01,Regular Air,46.71,8.69,...,Barry French,Nunavut,Nunavut,Consumer,Office Supplies,Binders and Binder Accessories,"Cardinal Slant-D� Ring Binder, Heavy Gauge Vinyl",Small Box,0.39,10/3/2012
3,80,483,7/10/2011,High,30,4965.7595,0.08,Regular Air,1198.97,195.99,...,Clay Rozendal,Nunavut,Nunavut,Corporate,Technology,Telephones and Communication,R380,Small Box,0.58,7/12/2011
4,85,515,8/28/2010,Not Specified,19,394.2700,0.08,Regular Air,30.94,21.78,...,Carlos Soltero,Nunavut,Nunavut,Consumer,Office Supplies,Appliances,Holmes HEPA Air Purifier,Medium Box,0.50,8/30/2010


## 2. Clean the data
- Parse dates.
- Drop rows without sales or profit.
- Clip discounts to a sane range [0, 1].
- Create calculated fields: profit margin, shipping cost ratio, lead time.

In [3]:
df["Order Date"] = pd.to_datetime(df["Order Date"], errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], errors="coerce")

before = len(df)
df = df.dropna(subset=["Sales", "Profit", "Order Date"])
df = df[df["Sales"] > 0]
df["Discount"] = df["Discount"].clip(lower=0, upper=1)
df["Shipping Cost"] = df["Shipping Cost"].fillna(0)

df["Profit Margin"] = df["Profit"] / df["Sales"]
df["Shipping Cost Ratio"] = df["Shipping Cost"] / df["Sales"]
df["Lead Time Days"] = (df["Ship Date"] - df["Order Date"]).dt.days
df["Order Year"] = df["Order Date"].dt.year
df["Order Month"] = df["Order Date"].dt.to_period("M").astype(str)

print(f"Dropped {before - len(df)} rows during cleaning. Remaining: {len(df):,}")

Dropped 0 rows during cleaning. Remaining: 8,399


## 3. Headline KPIs

In [4]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
profit_margin = total_profit / total_sales
avg_discount = df["Discount"].mean()
avg_lead_time = df["Lead Time Days"].mean()
shipping_ratio = df["Shipping Cost"].sum() / total_sales
orders = df["Order ID"].nunique()

print(f"Total sales:        ${total_sales:,.0f}")
print(f"Total profit:       ${total_profit:,.0f}")
print(f"Profit margin:      {profit_margin:.1%}")
print(f"Average discount:   {avg_discount:.1%}")
print(f"Avg lead time:      {avg_lead_time:.1f} days")
print(f"Shipping cost ratio:{shipping_ratio:.1%}")
print(f"Distinct orders:    {orders:,}")

Total sales:        $14,915,601
Total profit:       $1,521,768
Profit margin:      10.2%
Average discount:   5.0%
Avg lead time:      2.0 days
Shipping cost ratio:0.7%
Distinct orders:    5,496


## 4. Sales & profit by product category
Office Supplies dominates volume, but Technology carries the highest margin.

In [5]:
cat = (
    df.groupby("Product Category")
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"),
           orders=("Order ID", "nunique"))
      .reset_index()
)
cat["profit_margin_pct"] = (cat["total_profit"] / cat["total_sales"] * 100).round(2)
cat["avg_discount_pct"] = (cat["avg_discount"] * 100).round(2)
cat = cat.sort_values("total_sales", ascending=False)
cat[["total_sales", "total_profit"]] = cat[["total_sales", "total_profit"]].round(2)
cat

,Product Category,total_sales,total_profit,avg_discount,orders,profit_margin_pct,avg_discount_pct
2,Technology,5984248.18,886313.52,0.048746,1858,14.81,4.87
0,Furniture,5178590.54,117433.03,0.049287,1561,2.27,4.93
1,Office Supplies,3752762.10,518021.43,0.050230,3632,13.80,5.02


In [6]:
fig, ax = plt.subplots()
x = np.arange(len(cat))
w = 0.38
ax.bar(x - w / 2, cat["total_sales"], w, label="Sales", color=PALETTE[0])
ax.bar(x + w / 2, cat["total_profit"], w, label="Profit", color=PALETTE[3])
ax.set_xticks(x)
ax.set_xticklabels(cat["Product Category"])
ax.set_ylabel("USD")
ax.set_title("Sales vs Profit by Product Category")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "sales_profit_by_category.png", dpi=120)
plt.close(fig)

## 5. Profit by region
Some regions are profitable but thin; others are loss-making.

In [7]:
reg = (
    df.groupby("Region")
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           avg_discount=("Discount", "mean"))
      .reset_index()
)
reg["profit_margin_pct"] = (reg["total_profit"] / reg["total_sales"] * 100).round(2)
reg["avg_discount_pct"] = (reg["avg_discount"] * 100).round(2)
reg = reg.sort_values("total_profit", ascending=False)
reg[["total_sales", "total_profit"]] = reg[["total_sales", "total_profit"]].round(2)
reg

,Region,total_sales,total_profit,avg_discount,profit_margin_pct,avg_discount_pct
3,Ontario,3063212.48,346868.54,0.049146,11.32,4.91
4,Prarie,2837304.60,321160.12,0.049936,11.32,4.99
6,West,3597549.28,297008.61,0.048815,8.26,4.88
0,Atlantic,2014248.20,238960.66,0.050713,11.86,5.07
5,Quebec,1510195.08,140426.65,0.049808,9.30,4.98
1,Northwest Territories,800847.33,100653.08,0.050076,12.57,5.01
7,Yukon,975867.37,73849.21,0.050812,7.57,5.08
2,Nunavut,116376.48,2841.11,0.052278,2.44,5.23


In [8]:
fig, ax = plt.subplots()
colors = [PALETTE[0] if p >= 0 else PALETTE[3] for p in reg["total_profit"]]
ax.barh(reg["Region"], reg["total_profit"], color=colors)
ax.set_xlabel("Total profit (USD)")
ax.set_title("Profit by Region")
ax.invert_yaxis()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "profit_by_region.png", dpi=120)
plt.close(fig)

## 6. Discount impact on profitability
Discounts above ~15% systematically destroy margin.

In [9]:
bins = [-0.001, 0.0, 0.05, 0.10, 0.15, 0.20, 1.0]
labels = ["0%", "1-5%", "6-10%", "11-15%", "16-20%", "20%+"]
df["Discount Band"] = pd.cut(df["Discount"], bins=bins, labels=labels)

disc = (
    df.groupby("Discount Band", observed=True)
      .agg(total_sales=("Sales", "sum"),
           total_profit=("Profit", "sum"),
           line_items=("Sales", "size"))
      .reset_index()
)
disc["profit_margin_pct"] = (disc["total_profit"] / disc["total_sales"] * 100).round(2)
disc["total_sales"] = disc["total_sales"].round(2)
disc["total_profit"] = disc["total_profit"].round(2)
disc

,Discount Band,total_sales,total_profit,line_items,profit_margin_pct
0,0%,1493748.22,188188.77,756,12.60
1,1-5%,6932776.82,828183.46,3906,11.95
2,6-10%,6486807.52,505879.74,3732,7.80
3,11-15%,586.11,98.44,1,16.80
4,16-20%,679.86,-83.64,2,-12.30
5,20%+,1002.30,-498.79,2,-49.76


In [10]:
fig, ax = plt.subplots()
ax.bar(disc["Discount Band"].astype(str), disc["profit_margin_pct"],
       color=[PALETTE[0] if m >= 0 else PALETTE[3] for m in disc["profit_margin_pct"]])
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Profit margin (%)")
ax.set_xlabel("Discount band")
ax.set_title("Profit Margin by Discount Band")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "discount_impact.png", dpi=120)
plt.close(fig)

## 7. Shipping efficiency
Delivery Truck is cheapest per order but slower; Regular Air is fastest.

In [11]:
ship = (
    df.groupby("Ship Mode")
      .agg(line_items=("Sales", "size"),
           total_shipping_cost=("Shipping Cost", "sum"),
           avg_shipping_cost=("Shipping Cost", "mean"),
           avg_lead_time=("Lead Time Days", "mean"),
           total_sales=("Sales", "sum"))
      .reset_index()
)
ship["shipping_cost_pct_of_sales"] = (ship["total_shipping_cost"] / ship["total_sales"] * 100).round(2)
ship["avg_shipping_cost"] = ship["avg_shipping_cost"].round(2)
ship["avg_lead_time"] = ship["avg_lead_time"].round(1)
ship["total_shipping_cost"] = ship["total_shipping_cost"].round(2)
ship = ship.sort_values("avg_shipping_cost")
ship

,Ship Mode,line_items,total_shipping_cost,avg_shipping_cost,avg_lead_time,total_sales,shipping_cost_pct_of_sales
2,Regular Air,6270,48008.19,7.66,2.0,7506303.209,0.64
1,Express Air,983,7850.91,7.99,2.0,1184418.897,0.66
0,Delivery Truck,1146,51971.94,45.35,2.0,6224878.718,0.83


In [12]:
fig, ax = plt.subplots()
ax.scatter(ship["avg_lead_time"], ship["avg_shipping_cost"],
           s=ship["line_items"] / 5, color=PALETTE[1], alpha=0.8)
for _, r in ship.iterrows():
    ax.annotate(r["Ship Mode"], (r["avg_lead_time"], r["avg_shipping_cost"]),
                xytext=(6, 6), textcoords="offset points")
ax.set_xlabel("Average lead time (days)")
ax.set_ylabel("Average shipping cost (USD)")
ax.set_title("Shipping Efficiency: Cost vs Lead Time")
fig.tight_layout()
fig.savefig(OUT_CHARTS / "shipping_efficiency.png", dpi=120)
plt.close(fig)

## 8. Monthly sales trend

In [13]:
monthly = (
    df.groupby("Order Month")
      .agg(total_sales=("Sales", "sum"), total_profit=("Profit", "sum"))
      .reset_index()
      .sort_values("Order Month")
)
monthly["total_sales"] = monthly["total_sales"].round(2)
monthly["total_profit"] = monthly["total_profit"].round(2)

fig, ax = plt.subplots()
ax.plot(monthly["Order Month"], monthly["total_sales"], color=PALETTE[0], label="Sales")
ax.plot(monthly["Order Month"], monthly["total_profit"], color=PALETTE[3], label="Profit")
ax.set_xticks(monthly["Order Month"][::6])
ax.tick_params(axis="x", rotation=45)
ax.set_ylabel("USD")
ax.set_title("Monthly Sales and Profit Trend")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_CHARTS / "monthly_trend.png", dpi=120)
plt.close(fig)

## 9. Export dashboard data and summary
Every JSON file below is consumed by `docs/dashboard.json`.

In [14]:
cat.to_json(OUT_DOCS / "sales_profit_by_category.json", orient="records")
reg.to_json(OUT_DOCS / "profit_by_region.json", orient="records")
disc.to_json(OUT_DOCS / "discount_impact.json", orient="records")
ship.to_json(OUT_DOCS / "shipping_efficiency.json", orient="records")
monthly.to_json(OUT_DOCS / "monthly_trend.json", orient="records")

summary = {
    "total_sales": round(float(total_sales), 2),
    "total_profit": round(float(total_profit), 2),
    "profit_margin_pct": round(float(profit_margin * 100), 2),
    "avg_discount_pct": round(float(avg_discount * 100), 2),
    "avg_lead_time_days": round(float(avg_lead_time), 1),
    "shipping_cost_pct_of_sales": round(float(shipping_ratio * 100), 2),
    "orders": int(orders),
    "line_items": int(len(df)),
    "loss_making_subcategories": int(
        (df.groupby("Product Sub-Category")["Profit"].sum() < 0).sum()
    ),
}
with open(ROOT / "outputs" / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

{
  "total_sales": 14915600.82,
  "total_profit": 1521767.98,
  "profit_margin_pct": 10.2,
  "avg_discount_pct": 4.97,
  "avg_lead_time_days": 2.0,
  "shipping_cost_pct_of_sales": 0.72,
  "orders": 5496,
  "line_items": 8399,
  "loss_making_subcategories": 4
}


## 10. Conclusions
- **Technology** is the highest-margin category; **Office Supplies** drives volume.
- **Discounts above 15%** turn margin negative — cap them.
- **Delivery Truck** is the cheapest ship mode; reserve **Regular Air** for high-priority orders.
- Regional profitability varies widely — rebalance promotions toward profitable regions.